In [6]:
import torch 
import torch.nn.functional as F
import torch.nn as nn
import matplotlib.pyplot as plt

In [1]:
words = open('names.txt','r').read().splitlines()
len(words)
words[:10]

['emma',
 'olivia',
 'ava',
 'isabella',
 'sophia',
 'charlotte',
 'mia',
 'amelia',
 'harper',
 'evelyn']

In [7]:
chars =sorted(list(set(''.join(words))))
sti={s:i+1 for i,s in enumerate(chars)} 
sti['.']=0
its={s:i for i,s in sti.items()}
vocab_size=len(its)
its

{1: 'a',
 2: 'b',
 3: 'c',
 4: 'd',
 5: 'e',
 6: 'f',
 7: 'g',
 8: 'h',
 9: 'i',
 10: 'j',
 11: 'k',
 12: 'l',
 13: 'm',
 14: 'n',
 15: 'o',
 16: 'p',
 17: 'q',
 18: 'r',
 19: 's',
 20: 't',
 21: 'u',
 22: 'v',
 23: 'w',
 24: 'x',
 25: 'y',
 26: 'z',
 0: '.'}

In [9]:
import random
random.seed(42)
random.shuffle(words)

In [11]:
cont_size=8

def build_ds(words):
    X,Y=[],[]
    for w in words:
        context=[0]*cont_size
        #print(w)
        for ch in w+'.':
            ix=sti[ch]
            X.append(context)
            Y.append(ix)
            #print(''.join(its[i] for i in context),"------->",its[ix])
            context  = context[1:]+[ix]
    X=torch.tensor(X)
    Y=torch.tensor(Y)
    return X,Y
import random
random.seed(42)
random.shuffle(words)
n1=int(0.8*len(words))
n2=int(0.9*len(words))
Xtr,Ytr = build_ds(words[:n1])
Xva,Yva = build_ds(words[n1:n2])
Xte,Yte = build_ds(words[n2:])

In [12]:
for x,y in zip(Xtr[:20],Ytr[:20]):
    print(''.join(its[ix.item()] for ix in x),"--->",its[y.item()])

........ ---> e
.......e ---> b
......eb ---> r
.....ebr ---> i
....ebri ---> m
...ebrim ---> a
..ebrima ---> .
........ ---> h
.......h ---> i
......hi ---> l
.....hil ---> t
....hilt ---> o
...hilto ---> n
..hilton ---> .
........ ---> j
.......j ---> h
......jh ---> e
.....jhe ---> n
....jhen ---> e
...jhene ---> .


In [23]:

#------------------------------------------------------------------------#
class Linear:
    def __init__(self,fanin,fanout,bias=True):
        self.weight = torch.randn((fanin,fanout))/fanin**0.5
        self.bias = torch.zeros(fanout) if bias else None
        
    def __call__(self,x):
        self.out = x @ self.weight
        if self.bias is not None:
            self.out+=self.bias
        return self.out
        
    def parameters(self):
        return [self.weight]+([] if self.bias is None else [self.bias])
#------------------------------------------------------------------------#
class BatchNorm1d:
    def __init__(self,dim,eps=1e-5,momentum = 0.1):
        self.eps = eps
        self.momentum = momentum
        self.training = True
        #parameters(trained with backprop)
        self.gamma = torch.ones(dim)#gain
        self.beta = torch.zeros(dim)#bias
        #buffers
        self.running_mean = torch.zeros(dim)
        self.running_var = torch.ones(dim)


    def __call__(self,x):
        if self.training:
            if x.ndim ==2:
                dim =0
            elif x.ndim == 3:
                dim=(0,1)
            xmean = x.mean(dim,keepdim = True)
            xvar = x.var(dim,keepdim = True,unbiased=True)
        else:
            xmean = self.running_mean
            xvar = self.running_var
        xhat = (x-xmean) / torch.sqrt(xvar+self.eps)
        self.out = self.gamma*xhat + self.beta

        if self.training:
            with torch.no_grad():
                self.running_mean = (1-self.momentum) * self.running_mean + self.momentum * xmean
                self.running_var = (1-self.momentum) * self.running_var + self.momentum * xvar
        return self.out
    def parameters(self):
        return [self.gamma , self.beta]
#------------------------------------------------------------------------#
class Tanh:
    def __call__(self,x):
        self.out = torch.tanh(x)
        return self.out
    def parameters(self):
        return []
#------------------------------------------------------------------------#
class Embedding:
    def __init__(self, num_embeddings, embeddings_dim):
        self.weight = torch.randn((num_embeddings,embeddings_dim))

    def __call__(self, IX):
        self.out = self.weight[IX]
        return self.out

    def parameters(self):
        return[self.weight]
#------------------------------------------------------------------------#
class FlattenCons:
    def __init__(self , n):
        self.n=n
    def __call__(self,x):
        B,T,C = x.shape
        x = x.view(B,T//self.n,C*self.n)####flattening it into a row matrix for multiplication feasability
        if x.shape[1] == 1:
            x=x.squeeze(1)
        self.out = x
        return self.out

    def parameters(self):
        return []
#------------------------------------------------------------------------#

class Sequential:
    def __init__(self,layers):
        self.layers = layers

    def __call__(self,x):
        for layer in self.layers:
            x = layer(x)
        self.out = x
        return self.out

    def parameters(self):
        return [p for layer in self.layers for p in layer.parameters()]

In [25]:
torch.manual_seed(42);

In [31]:
#------------------------------------------------------------------------#
n_embd = 24
n_hidden = 128
model=Sequential([
    Embedding(vocab_size,n_embd),
    FlattenCons(2),Linear(n_embd * 2, n_hidden,bias = False),BatchNorm1d(n_hidden), Tanh(), 
    FlattenCons(2),Linear(n_hidden * 2, n_hidden,bias = False),BatchNorm1d(n_hidden), Tanh(),
    FlattenCons(2),Linear(n_hidden * 2, n_hidden,bias = False),BatchNorm1d(n_hidden), Tanh(),
    Linear(n_hidden,      vocab_size),
])
#------------------------------------------------------------------------#
with torch.no_grad():
    #layers[-1].weight*=0.1 #making last leyer less confident
    model.layers[-1].weight*=0.1
#------------------------------------------------------------------------#
parameters=model.parameters()
print("Total number of tunable parameters are",sum(p.nelement() for p in parameters))
for p in parameters:
    p.requires_grad = True

Total number of tunable parameters are 76579


In [34]:
max_steps = 200001
batch_size=32
lossi=[]

for i in range(max_steps):
    #constructing a mini batches so computation is easier
    ix = torch.randint(0,Xtr.shape[0],(batch_size,))
    Xe,Ye  = Xtr[ix],Ytr[ix]
    #forward pass
    #emb = C[Xe]
    #x = emb.view(emb.shape[0],-1)######################!!!!!!!!!!!!!!111
    logits = model(Xe)
    loss = F.cross_entropy(logits,Ye)
    
    for p in parameters:
        p.grad = None
    loss.backward()
    lr=0.1 if i < 150000 else 0.01 # enabling step wise gradient decay
    for p in model.parameters():
            p.data += -lr * p.grad  #gradient decay so just a small push for the model at the end

    #tracking
    if i  % 10000 == 0:
        print(f"{i:7d}/{max_steps-1:7d}: {loss.item():.4f}")
    lossi.append(loss.log10().item())


      0/ 200000: 3.3137
  10000/ 200000: 2.2840
  20000/ 200000: 2.3270
  30000/ 200000: 2.3151
  40000/ 200000: 2.1316
  50000/ 200000: 2.1044
  60000/ 200000: 2.4664
  70000/ 200000: 1.6600
  80000/ 200000: 1.9284
  90000/ 200000: 1.8985
 100000/ 200000: 2.0925
 110000/ 200000: 1.8332
 120000/ 200000: 2.1963
 130000/ 200000: 1.5761
 140000/ 200000: 1.9015
 150000/ 200000: 1.7868
 160000/ 200000: 1.7831
 170000/ 200000: 2.0707
 180000/ 200000: 1.8028
 190000/ 200000: 1.7302
 200000/ 200000: 1.6697


In [8]:
plt.plot(lossi)

NameError: name 'lossi' is not defined

In [ ]:
plt.plot(torch.tensor(lossi[:-1]).view(-1,1000).mean(1))

In [4]:
for layer in model.layers:
    layer.training = False

NameError: name 'model' is not defined

In [42]:
@torch.no_grad()
def split_loss(split):
    x,y = {
        "train":(Xtr,Ytr),
        "val":(Xva,Yva),
        "test":(Xte,Yte),
    }[split]
    logits = model(x)
    loss = F.cross_entropy(logits,y)
    print(split,loss.item())
split_loss("train")
split_loss("val")

train 1.7667491436004639
val 1.9926562309265137


In [44]:
#sample

for i in range(20):
    out=[]
    context = [0] * cont_size
    while True:
        logits = model(torch.tensor([context]))
        probs = F.softmax(logits , dim = 1)

        ix = torch.multinomial(probs, num_samples=1).item()

        context = context[1:] + [ix]
        out.append(ix)
        if ix==0:
            break
    print(''.join(its[i] for i in out))

fergi.
aliana.
channa.
meeya.
lovelle.
brenham.
daniko.
khilee.
jordy.
jeya.
leiky.
iman.
sanajla.
zoe.
belda.
graylynn.
elinorrose.
avehna.
aulaiah.
giuliana.
